In [1]:
# SimpleDirectoryReader is dynamic, detects file type and uses appropriate reader
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, Settings, PromptTemplate, get_response_synthesizer, Document
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.query_engine import RetrieverQueryEngine, TransformQueryEngine
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core.indices.query.query_transform import HyDEQueryTransform
from llama_index.core.postprocessor import SentenceTransformerRerank
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_parse import LlamaParse

from transformers import AutoTokenizer
from transformers import pipeline as hf_pipeline
from sentence_transformers import SentenceTransformer, util

import pandas as pd, re, ast, textwrap
from datasets import Dataset

from ragas.llms import llm_factory
from ragas.embeddings import embedding_factory
from ragas.metrics import answer_relevancy, faithfulness, context_recall, context_precision
from ragas import evaluate

from dotenv import load_dotenv, find_dotenv
from nanonets import NANONETSOCR

import torch
import os
import re
import csv

# --- For Azure ML Sandpit environment ---

# Project root path for Azure Sandpit environment
project_root_path = "/home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone" 

# # Change the current working directory to the project root
os.chdir(project_root_path)

dotenv_path = "/home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone/.env"

# --- For Local Development environment ---
# dotenv_path = find_dotenv()

# Load the .env file from the path that was found.
load_dotenv(dotenv_path=dotenv_path)

# Get the project root from the directory where the .env file was found.
project_root = os.path.dirname(dotenv_path)

# Get the relative directory name from the environment variable
relative_data_dir = os.getenv("VECTOR_DATASET_DIR")

# Create the full, absolute path by joining the project root with the relative name.
data_directory = os.path.join(project_root, relative_data_dir)

hf_token = os.getenv("HUGGINGFACE_TOKEN")
llama_cloud_api_key = os.getenv("LLAMA_CLOUD_API_KEY")
openai_api_key = os.getenv("OPENAI_API_KEY")

# 1. Initialize the Nanonets OCR client
nanonets_ocr = NANONETSOCR()
nanonets_api_key = os.getenv("NANONETS_API_KEY")
if not nanonets_api_key:
    raise ValueError("NANONETS_API_KEY not found in your .env file.")
nanonets_ocr.set_token(nanonets_api_key)


print(f"✅ Project root automatically determined as: {project_root}")
print(f"✅ .env file loaded from: {dotenv_path}")
print(f"📁 Data directory set to: {data_directory}")

✅ Project root automatically determined as: /home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone
✅ .env file loaded from: /home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone/.env
📁 Data directory set to: /home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone/Datasets/Vector_Dataset


## Vector/Graph Store Data Ingestion

In [ ]:
# --- Helper function to safely parse Nanonets response ---
# Extracts text from JSON response from Nanonets OCR API
def extract_text_from_nanonets_response(response, file_path):
    """
    A highly defensive function to safely extract text from a Nanonets OCR API response.
    It checks for different data structures to handle both single and multi-page documents.
    """
    filename = os.path.basename(file_path)

    # Check if response is a dictionary and contains 'results' key
    # results key holds a list and each item in the list is a JSON object from one file
    # JSON object acts as a container for the file's metadata and the extracted content by Nanonets
    if not isinstance(response, dict) or 'results' not in response or not isinstance(response['results'], list) or not response['results']:
        print(f" -> FAILED: {filename}. API response was malformed or empty. Response: {response}")
        return None

    first_result = response['results'][0]
    
    if not isinstance(first_result, dict) or 'page_data' not in first_result:
        print(f" -> FAILED: {filename}. 'results' object is malformed. Got: {first_result}")
        return None

    page_data_content = first_result['page_data']
    full_text = ""

    # Handle both list (multi-page) and dict (single-page) structures
    # If file is a multi-page document it iterates through each page in list and extracts the 'raw_text'
    # appending it to full_text with double newlines for separation
    if isinstance(page_data_content, list):
        for page in page_data_content:
            if isinstance(page, dict) and 'raw_text' in page:
                full_text += page['raw_text'] + "\\n\\n"
        if not full_text.strip():
            print(f" -> FAILED: {filename}. 'page_data' list contained no extractable text.")
            return None
    
    # If file is a single-page document it directly extracts the 'raw_text'
    elif isinstance(page_data_content, dict):
        if 'raw_text' in page_data_content:
            full_text = page_data_content['raw_text']
        else:
            print(f" -> FAILED: {filename}. 'page_data' dictionary missing 'raw_text' key.")
            return None
            
    else:
        print(f" -> FAILED: {filename}. 'page_data' has an unexpected type: {type(page_data_content)}.")
        return None

    return full_text.strip()

# Check which files need OCR processing
ocr_filepaths = []
other_filepaths = []
ocr_extensions = ['.pdf', '.jpg', '.jpeg', '.png']

for filename in os.listdir(data_directory):
    file_path = os.path.join(data_directory, filename)
    if os.path.isfile(file_path):
        if any(filename.lower().endswith(ext) for ext in ocr_extensions):
            ocr_filepaths.append(file_path)
        else:
            other_filepaths.append(file_path)

# --- Process OCR files with Nanonets OCR ---
all_documents = []

print(f"--- Processing {len(ocr_filepaths)} file(s) with Nanonets OCR. ---")
for file_path in ocr_filepaths:
    try:
        prediction = nanonets_ocr.convert_to_prediction(file_path)
        
        doc_text = extract_text_from_nanonets_response(prediction, file_path)
        
        # Only create a Document if text extraction was successful
        if doc_text:
            document = Document(text=doc_text, metadata={'file_path': file_path})
            all_documents.append(document)
            print(f" -> Successfully processed {os.path.basename(file_path)}")
            
    except Exception as e:
        print(f" -> FAILED during API call for {os.path.basename(file_path)}. An unexpected error occurred: {e}")

# --- Process other file types with SimpleDirectoryReader ---
if other_filepaths:
    print(f"\\n--- Loading {len(other_filepaths)} other file(s) with SimpleDirectoryReader. ---")
    try:
        other_docs = SimpleDirectoryReader(input_files=other_filepaths).load_data()
        all_documents.extend(other_docs)
        print(f" -> Successfully parsed {len(other_docs)} other document(s).")
    except Exception as e:
        print(f" -> FAILED to parse other files. Error: {e}")

documents = all_documents
print(f"\\n--- Ingestion complete ---")
print(f"Successfully loaded a total of {len(documents)} document(s).")



--- Processing 11 file(s) with Nanonets OCR. ---
 -> Successfully processed ANNUAL CRIME BRIEF 2020.pdf
 -> Successfully processed ANNUAL CRIME BRIEF 2021.pdf
 -> Successfully processed ANNUAL CRIME BRIEF 2022.pdf
 -> Successfully processed Annual Crime Brief 2023.pdf


## Initialise Models

In [ ]:
model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token)

llm = HuggingFaceLLM(
    model_name=model_name,
    tokenizer_name=model_name,
    device_map="auto",
    max_new_tokens=1024,
    model_kwargs={"token": hf_token, "torch_dtype": torch.bfloat16},
    generate_kwargs={
        "temperature": 0.1,
        "repetition_penalty": 1.2,
        "do_sample": True,
    }
)

print("Meta-Llama-3.1-8B-Instruct initialized for conversational responses.")

# Initialize HyDE query transformation
hyde_transform = HyDEQueryTransform(
    llm=llm,
    include_original=True
)

print("HyDE Query Transform initialized.")

# Initialize Re-ranker LLM
reranker = SentenceTransformerRerank(
    model="BAAI/bge-reranker-base",
    top_n=5  # Number of nodes to return after re-ranking
)

## Initialise Vector Embeddings

In [ ]:
Settings.llm = llm
Settings.embed_model = "local:BAAI/bge-small-en-v1.5"

print("Global settings configured with Llama 3.1 and bge-small embedding model.")

## Chunking

In [ ]:
node_parser = SentenceSplitter(chunk_size=1024, chunk_overlap=200)
pipeline = IngestionPipeline(transformations=[node_parser])
nodes = pipeline.run(documents=documents)

print(f"Total nodes created with custom chunking: {len(nodes)}")
print(f"Chunk size: {node_parser.chunk_size}, Chunk overlap: {node_parser.chunk_overlap}")

# Instantiate the BM25Retriever
retriever = BM25Retriever.from_defaults(
    nodes=nodes,
    similarity_top_k=10
)

print("\nBM25Retriever has been built successfully.")

## Retrieve Answer from Datastore

In [ ]:
custom_prompt = PromptTemplate(
    "Answer the question with exactly ONE short noun phrase taken from the context. "
    "Do not justify, explain, or add punctuation beyond a single period. "
    "Do NOT use markdown or LaTeX or code formatting of any kind.\n\n"
    "Context:\n{context_str}\n\n"
    "Question: {query_str}\n"
    "Answer:"
)

response_synthesizer = get_response_synthesizer(
    llm=llm,
    text_qa_template=custom_prompt
)

base_query_engine = RetrieverQueryEngine(
    retriever=retriever,
    response_synthesizer=response_synthesizer,
    node_postprocessors=[reranker],
)

# Applies HyDE transformation first before querying with the base query engine
advanced_query_engine = TransformQueryEngine(
    base_query_engine,
    query_transform=hyde_transform
)

query_text = "Which scam type had the highest number of cases in 2020?"
response = advanced_query_engine.query(query_text)

print(str(response))

## Transform Extracted Answer To Be Conversational

In [ ]:
# Normalize the extracted answer
# Eg "19,966." -> "19,966"
extracted_answer = str(response)
answer_core = extracted_answer.strip()
answer_core = re.sub(r"\.\s*$", "", answer_core).strip()

if not answer_core:
    print("Sorry, I couldn’t extract an answer from the context.")
else:
    REPHRASE_PROMPT = (
        "You are a precise assistant. Write exactly ONE conversational sentence that answers the question.\n"
        "Hard constraints:\n"
        "- Use the Extracted answer exactly once.\n"
        "- Do not add any other information not present in the Extracted answer or the Question.\n"
        "- Do not repeat yourself.\n"
        "- End with a single period.\n\n"
        f"Extracted answer: {answer_core}\n"
        f"Question: {query_text}\n"
        "Answer:"
    )

    # Generate a short rephrased sentence to avoid loops
    rephrase_raw = llm.complete(REPHRASE_PROMPT, max_new_tokens=32)

    # Collapse whitespace
    s = " ".join(str(rephrase_raw).strip().split())

    # Remove a potential leading echo of the bare answer (e.g., "19,966. The total ...")
    if answer_core:
        s = re.sub(rf"^\s*{re.escape(answer_core)}\.\s*", "", s).strip()

    # Ensure exactly one sentence
    idx = s.find(".")
    s = (s[: idx + 1] if idx != -1 else s + ".").strip()

    # Enforce inclusion of the extracted answer exactly once
    # If the model dropped the value, fall back to the minimal guaranteed version
    if answer_core not in s:
        s = f"{answer_core}."

    print(s)

## Benchmarking

In [ ]:
# # --- Config paths ---
# relative_benchmark_path = os.getenv("VECTOR_BENCHMARK_DATASET_DIR")
# if not relative_benchmark_path:
#     raise ValueError("VECTOR_BENCHMARK_DATASET_DIR not set in .env")
# BENCHMARK_FILE_PATH = os.path.join(project_root, relative_benchmark_path)
# OUTPUT_FILENAME = "adv_vector_benchmark_results.csv"
# OUTPUT_FILE_PATH = os.path.join(os.getcwd(), OUTPUT_FILENAME)

# # --- Load and Prepare Data ---
# if not os.path.isfile(BENCHMARK_FILE_PATH):
#     raise FileNotFoundError(f"Benchmark file not found at: {BENCHMARK_FILE_PATH}")

# # Load the benchmark CSV, automatically handling potential encoding errors
# try:
#     benchmark_df = pd.read_csv(BENCHMARK_FILE_PATH)
# except UnicodeDecodeError:
#     print("UTF-8 decoding failed. Retrying with 'latin1' encoding.")
#     benchmark_df = pd.read_csv(BENCHMARK_FILE_PATH, encoding='latin1')

# benchmark_df['Context'] = benchmark_df['gt_context']

# # --- CONFIG: Set the number of rows to test ---
# # Slice to first 3 rows for quick test
# # benchmark_df = benchmark_df.head(3)
# print(f"Loaded {len(benchmark_df)} question-answer pairs for evaluation.")


# # --- Unified Generation and Context Pinpointing Logic ---
# from sentence_transformers import SentenceTransformer, util

# # Load a sentence transformer model to find the most relevant context
# similarity_model = SentenceTransformer('BAAI/bge-small-en-v1.5')

# def get_response_and_pinpoint_context(question: str, retriever):
#     """
#     Retrieves context, generates a response, and identifies the single most relevant 
#     context chunk that was used for the answer.
#     """
#     # 1. Retrieve nodes
#     retrieved_nodes = retriever.retrieve(question)
    
#     # De-duplicate and clean chunks
#     seen = set()
#     cleaned_chunks = []
#     for node in retrieved_nodes:
#         content = node.get_content().replace("(1 sentence)", "").strip()
#         if content and content not in seen:
#             cleaned_chunks.append(content)
#             seen.add(content)
            
#     # Combine into a single context string for the LLM
#     exact_context_for_llm = "\n\n---\n\n".join(cleaned_chunks)

#     # 2. Use the same successful prompt template
#     prompt_template = PromptTemplate(
#         "You are a precise Q&A assistant. Use ONLY the context to answer the single question. "
#         "Do not invent or add anything not present in the context. Do not repeat the question. "
#         "Do not answer any other questions. Output exactly one sentence and then stop.\n\n"
#         "Context:\n"
#         "---------------------\n"
#         "{context_str}\n"
#         "---------------------\n\n"
#         "User's Question: {query_str}\n"
#         "Answer: "
#     )
#     final_prompt = prompt_template.format(context_str=exact_context_for_llm, query_str=question)
    
#     # 3. Generate response
#     raw_response = llm.complete(final_prompt, max_new_tokens=64)
    
#     # Clean the response
#     s = str(raw_response).strip()
#     parts = s.split(".")
#     prediction = (parts[0] + ".").strip() if parts else s
    
#     # 4. Pinpoint the most relevant context chunk
#     most_relevant_chunk = ""
#     if prediction and cleaned_chunks:
#         # Encode the generated answer and all context chunks
#         answer_embedding = similarity_model.encode(prediction, convert_to_tensor=True)
#         chunk_embeddings = similarity_model.encode(cleaned_chunks, convert_to_tensor=True)
        
#         # Compute cosine similarities
#         cosine_scores = util.cos_sim(answer_embedding, chunk_embeddings)
        
#         # Find the chunk with the highest score
#         best_chunk_index = cosine_scores.argmax()
#         most_relevant_chunk = cleaned_chunks[best_chunk_index]
    
#     return prediction, cleaned_chunks, most_relevant_chunk


# # --- Generate Predictions ---
# ragas_data = {"question": [], "answer": [], "contexts": [], "ground_truth": []}

# print("\nGenerating predictions and pinpointing context for each question...")
# benchmark_df['response'] = ''
# benchmark_df['retrieved_contexts'] = ''

# # --- This is the rephrasing logic, now as a function ---
# def make_conversational(extracted_answer: str, question: str) -> str:
#     """Takes a direct answer and makes it conversational using a prompt."""
#     answer_core = extracted_answer.strip().rstrip('.')
#     if not answer_core:
#         return "Sorry, I couldn’t extract an answer from the context."

#     REPHRASE_PROMPT = (
#         "You are a precise assistant. Write exactly ONE conversational sentence that answers the question.\\n"
#         "Hard constraints:\\n"
#         "- Use the Extracted answer exactly once.\\n"
#         "- Do not add any other information not present in the Extracted answer or the Question.\\n"
#         "- Do not repeat yourself.\\n"
#         "- End with a single period.\\n\\n"
#         f"Extracted answer: {answer_core}\\n"
#         f"Question: {question}\\n"
#         "Answer:"
#     )
    
#     rephrase_raw = llm.complete(REPHRASE_PROMPT, max_new_tokens=64)
    
#     # Clean up the rephrased output
#     s = " ".join(str(rephrase_raw).strip().split())
#     if answer_core:
#         s = re.sub(rf"^\\s*{re.escape(answer_core)}\\.\\s*", "", s).strip()
#     idx = s.find(".")
#     s = (s[: idx + 1] if idx != -1 else s + ".").strip()
#     if answer_core not in s:
#         s = f"{answer_core}." # Fallback
        
#     return s
# # --- End of rephrasing function ---


# for i, row in benchmark_df.iterrows():
#     question = row['question'].strip()
#     ground_truth = row['gt_answer'].strip()
    
#     # 1. Get the direct answer from the advanced query engine
#     response_obj = advanced_query_engine.query(question)
#     direct_answer = str(response_obj).strip()
    
#     # 2. Make the direct answer conversational
#     conversational_prediction = make_conversational(direct_answer, question)

#     # 3. Extract context for Ragas
#     all_retrieved_chunks = [node.get_content() for node in response_obj.source_nodes]
#     single_best_chunk = all_retrieved_chunks[0] if all_retrieved_chunks else ""

#     # Store the CONVERSATIONAL response and context in the DataFrame
#     benchmark_df.at[i, 'response'] = conversational_prediction
#     benchmark_df.at[i, 'retrieved_contexts'] = single_best_chunk
    
#     # Collect data for Ragas evaluation
#     ragas_data["question"].append(question)
#     ragas_data["answer"].append(conversational_prediction) # Use the conversational answer for evaluation
#     ragas_data["contexts"].append(all_retrieved_chunks)
#     ragas_data["ground_truth"].append(ground_truth)
    
#     print(f"Processed {i+1}/{len(benchmark_df)}")

# # --- Run Ragas Evaluation ---
# ragas_dataset = Dataset.from_dict(ragas_data)
# print(f"\nPrepared {len(ragas_dataset)} valid samples for Ragas evaluation.")

# judge_llm = llm_factory(model="gpt-4o") 
# judge_embeddings = embedding_factory(model="text-embedding-ada-002")

# metrics_to_evaluate = [
#     answer_relevancy,
#     faithfulness,
#     context_recall,
#     context_precision
# ]

# print("\nRunning Ragas evaluation...")
# result = evaluate(
#     dataset=ragas_dataset,
#     metrics=metrics_to_evaluate,
#     llm=judge_llm,
#     embeddings=judge_embeddings,
# )
# print("Ragas evaluation complete.")

# # --- Format and Save Final Results ---
# scores_df = result.to_pandas()

# # Merge scores back into the main DataFrame
# benchmark_df = benchmark_df.reset_index(drop=True)
# scores_df = scores_df.reset_index(drop=True)
# metric_cols = [m.name for m in metrics_to_evaluate]
# benchmark_df = benchmark_df.join(scores_df[metric_cols])

# # Ensure the desired column order
# final_columns = ['question', 'gt_answer', 'Context', 'response', 'retrieved_contexts'] + metric_cols
# benchmark_df = benchmark_df[final_columns]

# # Save the final results to a CSV file
# benchmark_df.to_csv(OUTPUT_FILE_PATH, index=False)
# print(f"\nDetailed results saved to: {OUTPUT_FILE_PATH}")

# # Print overall performance metrics
# print("\n--- Overall Ragas Performance Metrics (Averages) ---")
# print(scores_df[metric_cols].mean(numeric_only=True))
# print("----------------------------------------------------")


## Benchmarking without Query Rewriting

In [ ]:
# ---- BENCHMARKING WITHOUT QUERY REWRITING ----
# --- Config paths ---
relative_benchmark_path = os.getenv("VECTOR_BENCHMARK_DATASET_DIR")
if not relative_benchmark_path:
    raise ValueError("VECTOR_BENCHMARK_DATASET_DIR not set in .env")
BENCHMARK_FILE_PATH = os.path.join(project_root, relative_benchmark_path)
OUTPUT_FILENAME = "(w/o HyDe)vector_benchmark_results.csv"
OUTPUT_FILE_PATH = os.path.join(os.getcwd(), OUTPUT_FILENAME)

# --- Load and Prepare Data ---
if not os.path.isfile(BENCHMARK_FILE_PATH):
    raise FileNotFoundError(f"Benchmark file not found at: {BENCHMARK_FILE_PATH}")

# Load the benchmark CSV, automatically handling potential encoding errors
try:
    benchmark_df = pd.read_csv(BENCHMARK_FILE_PATH)
except UnicodeDecodeError:
    print("UTF-8 decoding failed. Retrying with 'latin1' encoding.")
    benchmark_df = pd.read_csv(BENCHMARK_FILE_PATH, encoding='latin1')

benchmark_df['Context'] = benchmark_df['gt_context']

# --- CONFIG: Set the number of rows to test ---
# Slice to first 3 rows for quick test
benchmark_df = benchmark_df.head(3)
print(f"Loaded {len(benchmark_df)} question-answer pairs for evaluation.")


# --- Unified Generation and Context Pinpointing Logic ---
from sentence_transformers import SentenceTransformer, util

# Load a sentence transformer model to find the most relevant context
similarity_model = SentenceTransformer('BAAI/bge-small-en-v1.5')

def get_response_and_pinpoint_context(question: str, retriever):
    """
    Retrieves context, generates a response, and identifies the single most relevant 
    context chunk that was used for the answer.
    """
    # 1. Retrieve nodes
    retrieved_nodes = retriever.retrieve(question)
    
    # De-duplicate and clean chunks
    seen = set()
    cleaned_chunks = []
    for node in retrieved_nodes:
        content = node.get_content().replace("(1 sentence)", "").strip()
        if content and content not in seen:
            cleaned_chunks.append(content)
            seen.add(content)
            
    # Combine into a single context string for the LLM
    exact_context_for_llm = "\n\n---\n\n".join(cleaned_chunks)

    # 2. Use the same successful prompt template
    prompt_template = PromptTemplate(
        "You are a precise Q&A assistant. Use ONLY the context to answer the single question. "
        "Do not invent or add anything not present in the context. Do not repeat the question. "
        "Do not answer any other questions. Output exactly one sentence and then stop.\n\n"
        "Context:\n"
        "---------------------\n"
        "{context_str}\n"
        "---------------------\n\n"
        "User's Question: {query_str}\n"
        "Answer: "
    )
    final_prompt = prompt_template.format(context_str=exact_context_for_llm, query_str=question)
    
    # 3. Generate response
    raw_response = llm.complete(final_prompt, max_new_tokens=64)
    
    # Clean the response
    s = str(raw_response).strip()
    parts = s.split(".")
    prediction = (parts[0] + ".").strip() if parts else s
    
    # 4. Pinpoint the most relevant context chunk
    most_relevant_chunk = ""
    if prediction and cleaned_chunks:
        # Encode the generated answer and all context chunks
        answer_embedding = similarity_model.encode(prediction, convert_to_tensor=True)
        chunk_embeddings = similarity_model.encode(cleaned_chunks, convert_to_tensor=True)
        
        # Compute cosine similarities
        cosine_scores = util.cos_sim(answer_embedding, chunk_embeddings)
        
        # Find the chunk with the highest score
        best_chunk_index = cosine_scores.argmax()
        most_relevant_chunk = cleaned_chunks[best_chunk_index]
    
    return prediction, cleaned_chunks, most_relevant_chunk


# --- Generate Predictions ---
ragas_data = {"question": [], "answer": [], "contexts": [], "ground_truth": []}

print("\nGenerating predictions and pinpointing context for each question...")
benchmark_df['response'] = ''
benchmark_df['retrieved_contexts'] = ''

# --- This is the rephrasing logic, now as a function ---
def make_conversational(extracted_answer: str, question: str) -> str:
    """Takes a direct answer and makes it conversational using a prompt."""
    answer_core = extracted_answer.strip().rstrip('.')
    if not answer_core:
        return "Sorry, I couldn’t extract an answer from the context."

    REPHRASE_PROMPT = (
        "You are a precise assistant. Write exactly ONE conversational sentence that answers the question.\\n"
        "Hard constraints:\\n"
        "- Use the Extracted answer exactly once.\\n"
        "- Do not add any other information not present in the Extracted answer or the Question.\\n"
        "- Do not repeat yourself.\\n"
        "- End with a single period.\\n\\n"
        f"Extracted answer: {answer_core}\\n"
        f"Question: {question}\\n"
        "Answer:"
    )
    
    rephrase_raw = llm.complete(REPHRASE_PROMPT, max_new_tokens=64)
    
    # Clean up the rephrased output
    s = " ".join(str(rephrase_raw).strip().split())
    if answer_core:
        s = re.sub(rf"^\\s*{re.escape(answer_core)}\\.\\s*", "", s).strip()
    idx = s.find(".")
    s = (s[: idx + 1] if idx != -1 else s + ".").strip()
    if answer_core not in s:
        s = f"{answer_core}." # Fallback
        
    return s
# --- End of rephrasing function ---


for i, row in benchmark_df.iterrows():
    question = row['question'].strip()
    ground_truth = row['gt_answer'].strip()
    
    # 1. Get the direct answer from the advanced query engine
    response = base_query_engine.query(question)
    direct_answer = str(response_obj).strip()
    
    # 2. Make the direct answer conversational
    conversational_prediction = make_conversational(direct_answer, question)

    # 3. Extract context for Ragas
    all_retrieved_chunks = [node.get_content() for node in response_obj.source_nodes]
    single_best_chunk = all_retrieved_chunks[0] if all_retrieved_chunks else ""

    # Store the CONVERSATIONAL response and context in the DataFrame
    benchmark_df.at[i, 'response'] = conversational_prediction
    benchmark_df.at[i, 'retrieved_contexts'] = single_best_chunk
    
    # Collect data for Ragas evaluation
    ragas_data["question"].append(question)
    ragas_data["answer"].append(conversational_prediction) # Use the conversational answer for evaluation
    ragas_data["contexts"].append(all_retrieved_chunks)
    ragas_data["ground_truth"].append(ground_truth)
    
    print(f"Processed {i+1}/{len(benchmark_df)}")

# --- Run Ragas Evaluation ---
ragas_dataset = Dataset.from_dict(ragas_data)
print(f"\nPrepared {len(ragas_dataset)} valid samples for Ragas evaluation.")

judge_llm = llm_factory(model="gpt-4o") 
judge_embeddings = embedding_factory(model="text-embedding-ada-002")

metrics_to_evaluate = [
    answer_relevancy,
    faithfulness,
    context_recall,
    context_precision
]

print("\nRunning Ragas evaluation...")
result = evaluate(
    dataset=ragas_dataset,
    metrics=metrics_to_evaluate,
    llm=judge_llm,
    embeddings=judge_embeddings,
)
print("Ragas evaluation complete.")

# --- Format and Save Final Results ---
scores_df = result.to_pandas()

# Merge scores back into the main DataFrame
benchmark_df = benchmark_df.reset_index(drop=True)
scores_df = scores_df.reset_index(drop=True)
metric_cols = [m.name for m in metrics_to_evaluate]
benchmark_df = benchmark_df.join(scores_df[metric_cols])

# Ensure the desired column order
final_columns = ['question', 'gt_answer', 'Context', 'response', 'retrieved_contexts'] + metric_cols
benchmark_df = benchmark_df[final_columns]

# Save the final results to a CSV file
benchmark_df.to_csv(OUTPUT_FILE_PATH, index=False)
print(f"\nDetailed results saved to: {OUTPUT_FILE_PATH}")

# Print overall performance metrics
print("\n--- Overall Ragas Performance Metrics (Averages) ---")
print(scores_df[metric_cols].mean(numeric_only=True))
print("----------------------------------------------------")


## Benchmarking without Reranker

In [ ]:
# ---- BENCHMARKING WITHOUT HyDE RERANKER ----
# --- Config paths ---
relative_benchmark_path = os.getenv("VECTOR_BENCHMARK_DATASET_DIR")
if not relative_benchmark_path:
    raise ValueError("VECTOR_BENCHMARK_DATASET_DIR not set in .env")
BENCHMARK_FILE_PATH = os.path.join(project_root, relative_benchmark_path)
OUTPUT_FILENAME = "(w/o reranker)ector_benchmark_results.csv"
OUTPUT_FILE_PATH = os.path.join(os.getcwd(), OUTPUT_FILENAME)

# --- Load and Prepare Data ---
if not os.path.isfile(BENCHMARK_FILE_PATH):
    raise FileNotFoundError(f"Benchmark file not found at: {BENCHMARK_FILE_PATH}")

# Load the benchmark CSV, automatically handling potential encoding errors
try:
    benchmark_df = pd.read_csv(BENCHMARK_FILE_PATH)
except UnicodeDecodeError:
    print("UTF-8 decoding failed. Retrying with 'latin1' encoding.")
    benchmark_df = pd.read_csv(BENCHMARK_FILE_PATH, encoding='latin1')

benchmark_df['Context'] = benchmark_df['gt_context']

# --- CONFIG: Set the number of rows to test ---
# Slice to first 3 rows for quick test
# benchmark_df = benchmark_df.head(3)
print(f"Loaded {len(benchmark_df)} question-answer pairs for evaluation.")


# --- Unified Generation and Context Pinpointing Logic ---
from sentence_transformers import SentenceTransformer, util

# Load a sentence transformer model to find the most relevant context
similarity_model = SentenceTransformer('BAAI/bge-small-en-v1.5')

def get_response_and_pinpoint_context(question: str, retriever):
    """
    Retrieves context, generates a response, and identifies the single most relevant 
    context chunk that was used for the answer.
    """
    # 1. Retrieve nodes
    retrieved_nodes = retriever.retrieve(question)
    
    # De-duplicate and clean chunks
    seen = set()
    cleaned_chunks = []
    for node in retrieved_nodes:
        content = node.get_content().replace("(1 sentence)", "").strip()
        if content and content not in seen:
            cleaned_chunks.append(content)
            seen.add(content)
            
    # Combine into a single context string for the LLM
    exact_context_for_llm = "\n\n---\n\n".join(cleaned_chunks)

    # 2. Use the same successful prompt template
    prompt_template = PromptTemplate(
        "You are a precise Q&A assistant. Use ONLY the context to answer the single question. "
        "Do not invent or add anything not present in the context. Do not repeat the question. "
        "Do not answer any other questions. Output exactly one sentence and then stop.\n\n"
        "Context:\n"
        "---------------------\n"
        "{context_str}\n"
        "---------------------\n\n"
        "User's Question: {query_str}\n"
        "Answer: "
    )
    final_prompt = prompt_template.format(context_str=exact_context_for_llm, query_str=question)
    
    # 3. Generate response
    raw_response = llm.complete(final_prompt, max_new_tokens=64)
    
    # Clean the response
    s = str(raw_response).strip()
    parts = s.split(".")
    prediction = (parts[0] + ".").strip() if parts else s
    
    # 4. Pinpoint the most relevant context chunk
    most_relevant_chunk = ""
    if prediction and cleaned_chunks:
        # Encode the generated answer and all context chunks
        answer_embedding = similarity_model.encode(prediction, convert_to_tensor=True)
        chunk_embeddings = similarity_model.encode(cleaned_chunks, convert_to_tensor=True)
        
        # Compute cosine similarities
        cosine_scores = util.cos_sim(answer_embedding, chunk_embeddings)
        
        # Find the chunk with the highest score
        best_chunk_index = cosine_scores.argmax()
        most_relevant_chunk = cleaned_chunks[best_chunk_index]
    
    return prediction, cleaned_chunks, most_relevant_chunk


# --- Generate Predictions ---
ragas_data = {"question": [], "answer": [], "contexts": [], "ground_truth": []}

print("\nGenerating predictions and pinpointing context for each question...")
benchmark_df['response'] = ''
benchmark_df['retrieved_contexts'] = ''

# --- This is the rephrasing logic, now as a function ---
def make_conversational(extracted_answer: str, question: str) -> str:
    """Takes a direct answer and makes it conversational using a prompt."""
    answer_core = extracted_answer.strip().rstrip('.')
    if not answer_core:
        return "Sorry, I couldn’t extract an answer from the context."

    REPHRASE_PROMPT = (
        "You are a precise assistant. Write exactly ONE conversational sentence that answers the question.\\n"
        "Hard constraints:\\n"
        "- Use the Extracted answer exactly once.\\n"
        "- Do not add any other information not present in the Extracted answer or the Question.\\n"
        "- Do not repeat yourself.\\n"
        "- End with a single period.\\n\\n"
        f"Extracted answer: {answer_core}\\n"
        f"Question: {question}\\n"
        "Answer:"
    )
    
    rephrase_raw = llm.complete(REPHRASE_PROMPT, max_new_tokens=64)
    
    # Clean up the rephrased output
    s = " ".join(str(rephrase_raw).strip().split())
    if answer_core:
        s = re.sub(rf"^\\s*{re.escape(answer_core)}\\.\\s*", "", s).strip()
    idx = s.find(".")
    s = (s[: idx + 1] if idx != -1 else s + ".").strip()
    if answer_core not in s:
        s = f"{answer_core}." # Fallback
        
    return s
# --- End of rephrasing function ---


for i, row in benchmark_df.iterrows():
    question = row['question'].strip()
    ground_truth = row['gt_answer'].strip()
    
    # 1. Get the direct answer from the advanced query engine
    response_obj = advanced_query_engine.query(question)
    direct_answer = str(response_obj).strip()
    
    # 2. Make the direct answer conversational
    conversational_prediction = make_conversational(direct_answer, question)

    # 3. Extract context for Ragas
    all_retrieved_chunks = [node.get_content() for node in response_obj.source_nodes]
    single_best_chunk = all_retrieved_chunks[0] if all_retrieved_chunks else ""

    # Store the CONVERSATIONAL response and context in the DataFrame
    benchmark_df.at[i, 'response'] = conversational_prediction
    benchmark_df.at[i, 'retrieved_contexts'] = single_best_chunk
    
    # Collect data for Ragas evaluation
    ragas_data["question"].append(question)
    ragas_data["answer"].append(conversational_prediction) # Use the conversational answer for evaluation
    ragas_data["contexts"].append(all_retrieved_chunks)
    ragas_data["ground_truth"].append(ground_truth)
    
    print(f"Processed {i+1}/{len(benchmark_df)}")

# --- Run Ragas Evaluation ---
ragas_dataset = Dataset.from_dict(ragas_data)
print(f"\nPrepared {len(ragas_dataset)} valid samples for Ragas evaluation.")

judge_llm = llm_factory(model="gpt-4o") 
judge_embeddings = embedding_factory(model="text-embedding-ada-002")

metrics_to_evaluate = [
    answer_relevancy,
    faithfulness,
    context_recall,
    context_precision
]

print("\nRunning Ragas evaluation...")
result = evaluate(
    dataset=ragas_dataset,
    metrics=metrics_to_evaluate,
    llm=judge_llm,
    embeddings=judge_embeddings,
)
print("Ragas evaluation complete.")

# --- Format and Save Final Results ---
scores_df = result.to_pandas()

# Merge scores back into the main DataFrame
benchmark_df = benchmark_df.reset_index(drop=True)
scores_df = scores_df.reset_index(drop=True)
metric_cols = [m.name for m in metrics_to_evaluate]
benchmark_df = benchmark_df.join(scores_df[metric_cols])

# Ensure the desired column order
final_columns = ['question', 'gt_answer', 'Context', 'response', 'retrieved_contexts'] + metric_cols
benchmark_df = benchmark_df[final_columns]

# Save the final results to a CSV file
benchmark_df.to_csv(OUTPUT_FILE_PATH, index=False)
print(f"\nDetailed results saved to: {OUTPUT_FILE_PATH}")

# Print overall performance metrics
print("\n--- Overall Ragas Performance Metrics (Averages) ---")
print(scores_df[metric_cols].mean(numeric_only=True))
print("----------------------------------------------------")
